In [0]:
# === Ingestion test setup ===
import sys
import os

# Actual path of YOUR repo on Databricks
repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

# Reload modules on each run (useful during development)
import importlib
if 'src.ingestion' in sys.modules:
    importlib.reload(sys.modules['src.ingestion'])

# Import ingestion functions
from src.ingestion import (
    generate_sample_dataset,
    ingest_csv_to_bronze,
    ingest_csv_to_bronze_memory,
    clean_column_name,
    create_spark_session,
)

# Spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("test-ingestion").getOrCreate()

print(f"✓ Spark version: {spark.version}")
print(f"✓ src.ingestion successfully imported")
print(f"✓ Available functions: generate_sample_dataset, ingest_csv_to_bronze, ingest_csv_to_bronze_memory, clean_column_name")

In [0]:
# === Generate a synthetic dataset of 10000 rows ===
raw_path = f"{repo_root}/data/raw/"

import os
os.makedirs(raw_path, exist_ok=True)

generate_sample_dataset(raw_path, n_rows=10000)

files = os.listdir(raw_path)
print(f"✓ Files in data/raw/: {files}")

In [0]:
# === Bronze ingestion test (memory-only version, Serverless-compatible) ===
# On Databricks Serverless, Spark cannot write files.
# Therefore we use the in-memory version that returns a DataFrame directly.
# Persistent Parquet storage will be set up in J7-J8 with Unity Catalog Volumes.

import importlib
if 'src.ingestion' in sys.modules:
    importlib.reload(sys.modules['src.ingestion'])

from src.ingestion import ingest_csv_to_bronze_memory

# Run the in-memory ingestion
df_bronze = ingest_csv_to_bronze_memory(spark, raw_path)

print(f"\n✓ Ingestion successful (in memory)")
print(f"✓ Returned object type: {type(df_bronze).__name__}")

In [0]:
# === Verification of the Bronze DataFrame in memory ===

print(f"✓ Total rows: {df_bronze.count()}")
print(f"✓ Columns ({len(df_bronze.columns)}): {df_bronze.columns}")

print("\n--- Preview (first 5 rows) ---")
df_bronze.show(5, truncate=False)

print("\n--- Schema ---")
df_bronze.printSchema()

print("\n--- Count by label ---")
from pyspark.sql import functions as F
df_bronze.groupBy("label").count().orderBy(F.col("count").desc()).show()

print("\n--- Metadata columns added ---")
df_bronze.select("_source_file", "_ingestion_ts").show(3, truncate=False)